# Bagian 4: Eksperimen dan Evaluasi

Notebook ini melakukan:
1. Forward propagation from scratch menggunakan arsitektur terbaik
2. Perbandingan Conv2D (shared) vs LocallyConnected2D (non-shared)
3. Analisis pengaruh hyperparameter
4. Evaluasi dan kesimpulan

## 1. Setup & Import

In [2]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import json
from datetime import datetime

# Import custom layers
from cnn.layers.conv2d import Conv2D
from cnn.layers.locallyconnected2d import LocallyConnected2D
from cnn.layers.pooling2d import MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from cnn.layers.flatten import Flatten
from cnn.layers.dense import Dense
from cnn.layers.activation import get_activation
from cnn.utility import batch_loader

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.21.0
GPU available: []


## 2. Load Training Results

In [4]:
# Load hasil pelatihan
results_df = pd.read_csv('cnn_hasil_pelatihan.csv')

# Parse filters dan kernel_size dari string
import ast
results_df['filters'] = results_df['filters'].apply(ast.literal_eval)
results_df['kernel_size'] = results_df['kernel_size'].apply(ast.literal_eval)

# Sort by F1 score
results_sorted = results_df.sort_values('f1_macro', ascending=False)

print("Top 5 Models by Macro F1-Score:")
print("="*80)
print(results_sorted[[
    'experiment_id', 'model_name', 'f1_macro', 'test_accuracy',
    'num_conv_layers', 'filters', 'kernel_size', 'pooling_type', 'num_params'
]].head())

# Best model
best_model_row = results_sorted.iloc[0]
best_model_name = best_model_row['model_name']
best_model_path = f'../models/cnn/{best_model_name}.h5'

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*80}")
print(f"F1 Score: {best_model_row['f1_macro']:.4f}")
print(f"Accuracy: {best_model_row['test_accuracy']:.4f}")
print(f"Layers: {best_model_row['num_conv_layers']}")
print(f"Filters: {best_model_row['filters']}")
print(f"Kernel: {best_model_row['kernel_size']}")
print(f"Pooling: {best_model_row['pooling_type']}")
print(f"Parameters: {best_model_row['num_params']:,}")
print(f"{'='*80}")

Top 5 Models by Macro F1-Score:
    experiment_id model_name  f1_macro  test_accuracy  num_conv_layers  \
14             15   model_15  0.819792       0.819000                3   
10             11   model_11  0.798122       0.797333                3   
15             16   model_16  0.792097       0.790667                3   
11             12   model_12  0.763133       0.763333                3   
12             13   model_13  0.758414       0.755667                3   

      filters kernel_size pooling_type  num_params  
14  [64, 128]      [5, 5]          max      636806  
10   [32, 64]      [5, 5]          max      165254  
15  [64, 128]      [5, 5]      average      636806  
11   [32, 64]      [5, 5]      average      165254  
12  [64, 128]      [3, 3]          max      240518  

BEST MODEL: model_15
F1 Score: 0.8198
Accuracy: 0.8190
Layers: 3
Filters: [64, 128]
Kernel: [5, 5]
Pooling: max
Parameters: 636,806


## 3. Load Dataset

In [5]:
# Dataset configuration
DATASET_PATH = Path('../dataset')
IMG_SIZE = (128, 128)
CHANNELS = 3
RANDOM_SEED = 42

def load_images_from_directory(directory, img_size=(128, 128), channels=3):
    directory = Path(directory)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory}")
    
    class_folders = sorted([d for d in directory.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_folders]
    
    all_image_paths = []
    all_labels = []
    
    for class_idx, class_folder in enumerate(class_folders):
        image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            image_paths.extend(list(class_folder.glob(ext)))
            image_paths.extend(list(class_folder.glob(ext.upper())))
        image_paths = sorted([str(p) for p in image_paths])
        all_image_paths.extend(image_paths)
        all_labels.extend([class_idx] * len(image_paths))
    
    images = batch_loader(all_image_paths, h=img_size[0], w=img_size[1], c=channels)
    labels = np.array(all_labels)
    
    return images, labels, class_names

# Load test set
test_dir = DATASET_PATH / 'seg_test' / 'seg_test'
print("Loading test set...")
X_test, y_test, class_names = load_images_from_directory(test_dir, IMG_SIZE, CHANNELS)
print(f"Test set shape: {X_test.shape}")
print(f"Classes: {class_names}")

Loading test set...
Test set shape: (6000, 128, 128, 3)
Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


## 4. Load Best Keras Model

In [6]:
# Load best model
keras_model = keras.models.load_model(best_model_path)
print(f"Loaded model: {best_model_path}")
print("\nModel Summary:")
keras_model.summary()

# Evaluate Keras model
print("\nEvaluating Keras model on test set...")
keras_loss, keras_acc = keras_model.evaluate(X_test, y_test, verbose=0)
keras_pred = keras_model.predict(X_test, verbose=0)
keras_pred_classes = np.argmax(keras_pred, axis=1)
keras_f1 = f1_score(y_test, keras_pred_classes, average='macro')

print(f"\nKeras Model Performance:")
print(f"  Accuracy: {keras_acc:.4f}")
print(f"  Loss: {keras_loss:.4f}")
print(f"  Macro F1: {keras_f1:.4f}")

Loaded model: ../models/cnn/model_15.h5

Model Summary:


Model: "cnn_3layers"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_1 (Conv2D)                 │ (None, 150, 150, 64)   │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv2D)                 │ (None, 75, 75, 128)    │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_3 (Conv2D)                 │ (None, 37, 37, 128)    │       409,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_3 (MaxPooling2D)        │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_pool                     │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 636,808 (2.43 MB)

 Trainable params: 636,806 (2.43 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)


Evaluating Keras model on test set...

Keras Model Performance:
  Accuracy: 0.7667
  Loss: 0.6286
  Macro F1: 0.7705


## 5. Build Custom Model (From Scratch - Shared Parameters)

In [7]:
class CustomCNN:
    """CNN from scratch using custom layers"""
    
    def __init__(self, keras_model):
        self.layers = []
        self.layer_names = []
        
        # Extract layers from Keras model
        for layer in keras_model.layers:
            layer_name = layer.name
            layer_type = type(layer).__name__
            
            print(f"Loading layer: {layer_name} ({layer_type})")
            
            if 'conv' in layer_name:
                custom_layer = Conv2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'maxpool' in layer_name:
                custom_layer = MaxPooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'avgpool' in layer_name:
                custom_layer = AveragePooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'global' in layer_name:
                custom_layer = GlobalAveragePooling2D(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'dense' in layer_name or 'output' in layer_name:
                custom_layer = Dense(layer)
                self.layers.append(custom_layer)
                self.layer_names.append(layer_name)
                
            elif 'dropout' in layer_name:
                # Skip dropout during inference
                print(f"  Skipping dropout layer (inference mode)")
                continue
            
            else:
                print(f"  Warning: Unknown layer type {layer_type}")
        
        print(f"\nTotal custom layers loaded: {len(self.layers)}")
    
    def forward(self, x):
        """Forward propagation"""
        for i, (layer, name) in enumerate(zip(self.layers, self.layer_names)):
            x = layer.forward(x)
            if i == 0:  # Only print first layer details
                print(f"  Layer {i+1} ({name}): output shape = {x.shape}")
        return x
    
    def predict(self, X, batch_size=32, verbose=True):
        """Predict on batch of images"""
        n_samples = X.shape[0]
        predictions = []
        
        for i in range(0, n_samples, batch_size):
            batch = X[i:i+batch_size]
            if verbose and i == 0:
                print(f"\nProcessing batch 1/{(n_samples-1)//batch_size + 1}")
                print(f"  Input shape: {batch.shape}")
            
            batch_pred = self.forward(batch)
            predictions.append(batch_pred)
        
        return np.vstack(predictions)

# Build custom model
print("Building custom CNN from scratch...")
print("="*80)
custom_model_shared = CustomCNN(keras_model)
print("="*80)

Building custom CNN from scratch...
Loading layer: conv_1 (Conv2D)
Loading layer: maxpool_1 (MaxPooling2D)
Loading layer: conv_2 (Conv2D)
Loading layer: maxpool_2 (MaxPooling2D)
Loading layer: conv_3 (Conv2D)
Loading layer: maxpool_3 (MaxPooling2D)
Loading layer: global_pool (GlobalAveragePooling2D)
Loading layer: dense_1 (Dense)
Loading layer: dropout (Dropout)
  Skipping dropout layer (inference mode)
Loading layer: output (Dense)

Total custom layers loaded: 9


## 6. Test Custom Model (Shared Parameters)

In [ ]:
print("\nTesting on test set...")

custom_pred_full = custom_model_shared.predict(X_test, batch_size=32, verbose=False)
custom_classes_full = np.argmax(custom_pred_full, axis=1)
custom_f1 = f1_score(y_test, custom_classes_full, average='macro')
custom_acc = np.mean(custom_classes_full == y_test)

print("\nCustom Model (Shared) Performance:")
print(f"  Accuracy: {custom_acc:.4f}")
print(f"  Macro F1: {custom_f1:.4f}")

print("\n" + "="*80)
print("KERAS vs CUSTOM (Shared) COMPARISON")
print("="*80)
print(f"Keras    - Accuracy: {keras_acc:.4f}, F1: {keras_f1:.4f}")
print(f"Custom   - Accuracy: {custom_acc:.4f}, F1: {custom_f1:.4f}")
print(f"Difference - Accuracy: {abs(keras_acc - custom_acc):.6f}, F1: {abs(keras_f1 - custom_f1):.6f}")
print("="*80)

## 7. Hyperparameter Analysis

In [ ]:
# Analyze effect of each hyperparameter
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Number of layers
results_df.boxplot(column='f1_macro', by='num_conv_layers', ax=axes[0, 0])
axes[0, 0].set_title('F1 Score by Number of Conv Layers')
axes[0, 0].set_xlabel('Number of Conv Layers')
axes[0, 0].set_ylabel('Macro F1 Score')
axes[0, 0].get_figure().suptitle('')

# 2. Kernel size
results_df['kernel_size_str'] = results_df['kernel_size'].astype(str)
results_df.boxplot(column='f1_macro', by='kernel_size_str', ax=axes[0, 1])
axes[0, 1].set_title('F1 Score by Kernel Size')
axes[0, 1].set_xlabel('Kernel Size')
axes[0, 1].set_ylabel('Macro F1 Score')
axes[0, 1].get_figure().suptitle('')

# 3. Pooling type
results_df.boxplot(column='f1_macro', by='pooling_type', ax=axes[1, 0])
axes[1, 0].set_title('F1 Score by Pooling Type')
axes[1, 0].set_xlabel('Pooling Type')
axes[1, 0].set_ylabel('Macro F1 Score')
axes[1, 0].get_figure().suptitle('')

# 4. Number of filters
results_df['filters_str'] = results_df['filters'].astype(str)
results_df.boxplot(column='f1_macro', by='filters_str', ax=axes[1, 1])
axes[1, 1].set_title('F1 Score by Filter Configuration')
axes[1, 1].set_xlabel('Filters')
axes[1, 1].set_ylabel('Macro F1 Score')
axes[1, 1].get_figure().suptitle('')

plt.tight_layout()
plt.savefig('../results/hyperparameter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("HYPERPARAMETER ANALYSIS")
print("="*80)

print("\n1. Number of Conv Layers:")
print(results_df.groupby('num_conv_layers')['f1_macro'].agg(['mean', 'std', 'min', 'max']))

print("\n2. Kernel Size:")
print(results_df.groupby('kernel_size_str')['f1_macro'].agg(['mean', 'std', 'min', 'max']))

print("\n3. Pooling Type:")
print(results_df.groupby('pooling_type')['f1_macro'].agg(['mean', 'std', 'min', 'max']))

print("\n4. Filter Configuration:")
print(results_df.groupby('filters_str')['f1_macro'].agg(['mean', 'std', 'min', 'max']))

## 8. Confusion Matrix

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Keras (Shared)
cm_keras = confusion_matrix(y_test, keras_pred_classes)
sns.heatmap(cm_keras, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title(f'Keras (Shared)\nF1: {keras_f1:.4f}')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Custom (Shared)
cm_custom = confusion_matrix(y_test, custom_classes_full)
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title(f'Custom (Shared)\nF1: {custom_f1:.4f}')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Classification Report

In [ ]:
print("\n" + "="*80)
print("CLASSIFICATION REPORT - KERAS (SHARED)")
print("="*80)
print(classification_report(y_test, keras_pred_classes, target_names=class_names))

print("\n" + "="*80)
print("CLASSIFICATION REPORT - CUSTOM (SHARED)")
print("="*80)
print(classification_report(y_test, custom_classes_full, target_names=class_names))